# Geometric, Negative Binomial & Hypergeometric Distributions

These distributions model **waiting times** and **sampling without replacement**.

## Geometric Distribution

| Property | Value |
|----------|-------|
| **Notation** | $X \sim \text{Geo}(p)$ |
| **Description** | Number of trials until first success |
| **Parameter** | $p$ = probability of success per trial |
| **Support** | $\{1, 2, 3, \ldots\}$ |
| **PMF** | $P(X = k) = (1-p)^{k-1} p$ |
| **Expectation** | $E[X] = 1/p$ |
| **Variance** | $\text{Var}(X) = (1-p)/p^2$ |

## Negative Binomial Distribution

| Property | Value |
|----------|-------|
| **Notation** | $X \sim \text{NegBin}(r, p)$ |
| **Description** | Number of trials until $r$-th success |
| **Parameters** | $r$ = target successes, $p$ = success probability |
| **Support** | $\{r, r+1, r+2, \ldots\}$ |
| **PMF** | $P(X = k) = \binom{k-1}{r-1} p^r (1-p)^{k-r}$ |
| **Expectation** | $E[X] = r/p$ |
| **Variance** | $\text{Var}(X) = r(1-p)/p^2$ |

## Hypergeometric Distribution

| Property | Value |
|----------|-------|
| **Notation** | $X \sim \text{HGeom}(N, K, n)$ |
| **Description** | Number of successes when sampling **without replacement** |
| **Parameters** | $N$ = population size, $K$ = success states in population, $n$ = draws |
| **Support** | $\{\max(0, n+K-N), \ldots, \min(n, K)\}$ |
| **PMF** | $P(X = k) = \frac{\binom{K}{k}\binom{N-K}{n-k}}{\binom{N}{n}}$ |
| **Expectation** | $E[X] = n\frac{K}{N}$ |
| **Variance** | $\text{Var}(X) = n\frac{K}{N}\frac{N-K}{N}\frac{N-n}{N-1}$ |

**Key insights:**
- $\text{Geo}(p) = \text{NegBin}(1, p)$
- Hypergeometric is like Binomial but **without replacement** (dependent trials)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

print("Libraries loaded!")

---
## 1. Geometric PMF — Waiting for First Success

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

p_values = [0.1, 0.3, 0.7]
colors = ['#e74c3c', '#2ecc71', '#3498db']

for ax, p, color in zip(axes, p_values, colors):
    rv = stats.geom(p)
    x = np.arange(1, int(3/p) + 5)
    ax.bar(x, rv.pmf(x), color=color, edgecolor='black', linewidth=0.8, alpha=0.8)
    ax.axvline(rv.mean(), color='black', linewidth=2, linestyle='--',
               label=f'E[X] = {rv.mean():.1f}')
    ax.set_title(f'Geo(p={p})\nVar = {rv.var():.2f}', fontsize=12, fontweight='bold')
    ax.set_xlabel('k (trial of first success)')
    ax.set_ylabel('P(X = k)')
    ax.legend(fontsize=10)

plt.suptitle('Geometric: Lower p → Longer Wait', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 2. Memoryless Property

$$P(X > s + t \mid X > s) = P(X > t)$$

"Given you've already waited $s$ trials without success, the remaining wait has the same distribution."
The Geometric is the **only** discrete distribution with this property.

In [ ]:
# Demonstrate memoryless property
np.random.seed(42)
p = 0.2
n_sims = 100000

# Simulate Geo(0.2)
samples = stats.geom.rvs(p, size=n_sims)

# P(X > 10)
p_gt_10 = np.mean(samples > 10)

# P(X > 15 | X > 5) should equal P(X > 10) by memoryless property
samples_gt_5 = samples[samples > 5]
p_gt_15_given_gt_5 = np.mean(samples_gt_5 > 15)

# From theory
p_gt_10_theory = (1 - p) ** 10

print("Memoryless Property: P(X > s+t | X > s) = P(X > t)")
print(f"  P(X > 10)            = {p_gt_10:.4f}  (theory: {p_gt_10_theory:.4f})")
print(f"  P(X > 15 | X > 5)   = {p_gt_15_given_gt_5:.4f}  (should ≈ P(X > 10))")
print(f"  Difference: {abs(p_gt_10 - p_gt_15_given_gt_5):.4f}")

# Visual: compare survival curves
fig, ax = plt.subplots(figsize=(10, 5))
k = np.arange(1, 40)
survival = (1 - p) ** k  # P(X > k)
ax.plot(k, survival, 'b-', linewidth=2.5, label='P(X > k) from start')

# Conditional: P(X > k+5 | X > 5)
k_cond = np.arange(1, 35)
ax.plot(k_cond, survival[:len(k_cond)], 'r--', linewidth=2.5,
        label='P(X > k+5 | X > 5) (shifted)')
ax.set_xlabel('k (additional trials)', fontsize=12)
ax.set_ylabel('Probability', fontsize=12)
ax.set_title('Memoryless Property: Same Decay Rate', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

---
## 3. Negative Binomial — Waiting for $r$-th Success

In [ ]:
# NegBin: number of failures before r successes (scipy convention)
# scipy.stats.nbinom(n=r, p=p) gives # failures before r successes
# We'll adjust for our convention (total trials)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

r_values = [1, 3, 10]
p = 0.3
colors = ['#e74c3c', '#2ecc71', '#3498db']

for ax, r, color in zip(axes, r_values, colors):
    # scipy nbinom gives failures, we want total trials = failures + r
    rv_failures = stats.nbinom(r, p)
    x_failures = np.arange(0, int(r/p + 4*np.sqrt(r*(1-p)/p**2)) + 1)
    x_trials = x_failures + r  # total trials = failures + successes
    
    ax.bar(x_trials, rv_failures.pmf(x_failures), color=color,
           edgecolor='black', linewidth=0.8, alpha=0.8)
    mean_trials = r / p
    ax.axvline(mean_trials, color='black', linewidth=2, linestyle='--',
               label=f'E[X] = r/p = {mean_trials:.1f}')
    ax.set_title(f'NegBin(r={r}, p={p})', fontsize=12, fontweight='bold')
    ax.set_xlabel('k (total trials)')
    ax.set_ylabel('P(X = k)')
    ax.legend(fontsize=10)

plt.suptitle('Negative Binomial: Trials Until r-th Success', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Negative Binomial = sum of r independent Geometric random variables")
print(f"Check: NegBin(1, p) = Geo(p) ✓")

---
## 4. Python: Working with Geometric & Negative Binomial

In [ ]:
from scipy.stats import geom, nbinom

p = 0.3

# Geometric
rv = geom(p)
print(f"Geometric(p={p}):")
print(f"  E[X]      = {rv.mean():.4f}  (1/p = {1/p:.4f})")
print(f"  Var(X)    = {rv.var():.4f}  ((1-p)/p² = {(1-p)/p**2:.4f})")
print(f"  P(X = 1)  = {rv.pmf(1):.4f}  (= p)")
print(f"  P(X ≤ 5)  = {rv.cdf(5):.4f}")
print(f"  Samples:    {rv.rvs(size=10, random_state=42)}")

# Negative Binomial (scipy: counts failures before r successes)
r = 3
rv_nb = nbinom(r, p)
print(f"\nNegBin(r={r}, p={p}) — scipy counts FAILURES:")
print(f"  E[failures]  = {rv_nb.mean():.4f}  (r(1-p)/p = {r*(1-p)/p:.4f})")
print(f"  E[trials]    = {rv_nb.mean() + r:.4f}  (r/p = {r/p:.4f})")
print(f"  Var(failures)= {rv_nb.var():.4f}")
print(f"  Samples (failures): {rv_nb.rvs(size=10, random_state=42)}")

---
## 5. Hypergeometric Distribution — Sampling Without Replacement

The **Hypergeometric** distribution answers: if you draw $n$ items from a population of $N$ (containing $K$ "successes"), how many successes do you get?

**Unlike Binomial**: trials are **dependent** — each draw changes the composition of the remaining pool.

$$P(X = k) = \frac{\binom{K}{k}\binom{N-K}{n-k}}{\binom{N}{n}}$$

**Intuition:** Choose $k$ of the $K$ successes **AND** $n-k$ of the $N-K$ failures, divided by total ways to choose $n$ items.

In [ ]:
# Classic example: Drawing cards from a deck
# N=52 cards, K=13 hearts, n=5 cards drawn
from math import comb

N, K, n = 52, 13, 5
rv = stats.hypergeom(N, K, n)

print(f"Draw {n} cards from a deck of {N}. X = number of hearts (K={K}).")
print(f"E[X] = n × K/N = {n} × {K}/{N} = {n * K / N:.4f}")
print(f"Var(X) = {rv.var():.4f}")
print()

# PMF for each possible value
print(f"{'k':>3s}  {'P(X=k)':>12s}  Formula")
print("-" * 55)
for k in range(min(n, K) + 1):
    if comb(N - K, n - k) == 0:
        continue
    p = rv.pmf(k)
    print(f"{k:3d}  {p:12.6f}  C({K},{k})·C({N-K},{n-k}) / C({N},{n})"
          f" = {comb(K,k)}·{comb(N-K,n-k)} / {comb(N,n)}")

print(f"\nSum of PMF = {sum(rv.pmf(k) for k in range(n+1)):.6f} ✓")

# Visualize
fig, ax = plt.subplots(figsize=(10, 5))
k_vals = np.arange(0, min(n, K) + 1)
ax.bar(k_vals, rv.pmf(k_vals), color='#9b59b6', edgecolor='black', linewidth=1.2, width=0.6)
ax.axvline(rv.mean(), color='red', linewidth=2, linestyle='--', label=f'E[X] = {rv.mean():.2f}')
ax.set_xlabel('k (hearts drawn)', fontsize=12)
ax.set_ylabel('P(X = k)', fontsize=12)
ax.set_title(f'Hypergeometric(N={N}, K={K}, n={n}): Hearts in a 5-Card Hand', fontsize=13, fontweight='bold')
ax.set_xticks(k_vals)
for ki in k_vals:
    ax.text(ki, rv.pmf(ki) + 0.005, f'{rv.pmf(ki):.4f}', ha='center', fontsize=10)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

---
### Hypergeometric vs Binomial

| | Hypergeometric | Binomial |
|---|---|---|
| **Sampling** | Without replacement | With replacement |
| **Trials** | Dependent | Independent |
| **P(success) per trial** | Changes after each draw | Constant $p$ |
| **When equivalent?** | $N \to \infty$ with $K/N = p$ | Always |

As the population $N$ grows large relative to the sample $n$, removing one item barely changes the remaining composition, so **HGeom → Binomial**.

In [ ]:
# Compare Hypergeometric vs Binomial as N grows
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Fixed: K/N = 0.25 (proportion of successes), n = 10
# Increase N: 40, 200, 2000
proportion = 0.25
n_draw = 10

for ax, N_pop in zip(axes, [40, 200, 2000]):
    K_pop = int(N_pop * proportion)
    rv_hg = stats.hypergeom(N_pop, K_pop, n_draw)
    rv_bin = stats.binom(n_draw, proportion)
    
    k = np.arange(0, n_draw + 1)
    ax.bar(k - 0.2, rv_hg.pmf(k), 0.4, color='#9b59b6', edgecolor='black',
           alpha=0.7, label=f'HGeom({N_pop}, {K_pop}, {n_draw})')
    ax.bar(k + 0.2, rv_bin.pmf(k), 0.4, color='#f39c12', edgecolor='black',
           alpha=0.7, label=f'Bin({n_draw}, {proportion})')
    ax.set_title(f'N = {N_pop}', fontsize=12, fontweight='bold')
    ax.set_xlabel('k (successes)')
    ax.set_ylabel('P(X = k)')
    ax.legend(fontsize=9)

plt.suptitle('Hypergeometric → Binomial as Population Grows',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Variance comparison (HGeom has correction factor (N-n)/(N-1)):")
for N_pop in [40, 200, 2000, 100000]:
    K_pop = int(N_pop * proportion)
    rv_hg = stats.hypergeom(N_pop, K_pop, n_draw)
    rv_bin = stats.binom(n_draw, proportion)
    correction = (N_pop - n_draw) / (N_pop - 1)
    print(f"  N={N_pop:>6d}: Var(HGeom) = {rv_hg.var():.4f}, "
          f"Var(Bin) = {rv_bin.var():.4f}, "
          f"correction = {correction:.4f}")

---
### Practical Examples

---
## Summary

| Distribution | PMF | $E[X]$ | $\text{Var}(X)$ |
|---|---|---|---|
| Geo($p$) | $(1-p)^{k-1}p$ | $1/p$ | $(1-p)/p^2$ |
| NegBin($r, p$) | $\binom{k-1}{r-1}p^r(1-p)^{k-r}$ | $r/p$ | $r(1-p)/p^2$ |
| HGeom($N, K, n$) | $\frac{\binom{K}{k}\binom{N-K}{n-k}}{\binom{N}{n}}$ | $n\frac{K}{N}$ | $n\frac{K}{N}\frac{N-K}{N}\frac{N-n}{N-1}$ |

| Key Facts | |
|-----------|---|
| Relationship | NegBin($r,p$) = sum of $r$ ind. Geo($p$) |
| Memoryless | Only Geometric has this (among discrete) |
| HGeom vs Binomial | Without replacement → HGeom; with replacement → Binomial |
| HGeom → Binomial | As $N \to \infty$ with $K/N = p$, HGeom$(N,K,n) \to$ Bin$(n,p)$ |

**Next:** Categorical Distribution — non-numeric outcomes.